# SEBI Investor Survey 2025 — Data Inspection

This notebook is the starting point for analysing the **SEBI Investor Survey 2025**, a study of investor behaviour conducted by SEBI. It is not an INDmoney app-usage or conversion dataset.

Official source: see [`docs/data_sources.md`](../docs/data_sources.md).

**Purpose of this notebook:** confirm the Python environment is working, then inspect the raw SEBI workbook(s) placed in `data/raw/` — file structure, sheet names, and column definitions — before any cleaning or analysis is written. Nothing about the source file's structure is assumed in advance.

**Planned workflow:** original data (`data/raw/`) → inspection and analysis (this notebook and future ones) → aggregate exports (`scripts/`, written to `data/processed/` and reviewed aggregates to `public/data/`) → Next.js dashboard.

## Environment check

Confirms the notebook is running inside the project's `.venv` kernel and that the required packages import correctly.

In [1]:
import sys
import platform
import importlib.metadata

import pandas as pd
import openpyxl
import python_calamine
import matplotlib

print(f"Python executable: {sys.executable}")
print(f"Python version:    {platform.python_version()}")
print(f"pandas:             {pd.__version__}")
print(f"openpyxl:           {openpyxl.__version__}")
print(f"python_calamine:    {importlib.metadata.version('python_calamine')}")
print(f"matplotlib:         {matplotlib.__version__}")

Python executable: /Users/keshavdubey/Downloads/Work/Monsoon 2026/Market Research & Validation/Mutual Funds Analysis/.venv/bin/python
Python version:    3.12.12
pandas:             3.0.5
openpyxl:           3.1.5
python_calamine:    0.8.2
matplotlib:         3.11.2


## 1. Inspect the files

List what is actually in `data/raw/` and, before loading anything, list the sheet names in both workbooks. `python_calamine` is used to read `.xlsx` because it is dramatically faster than `openpyxl` for the ~171 MB respondent workbook (whole-sheet load in ~10-15s vs. minutes), while still returning a plain list-of-lists we control entirely.

In [2]:
import os
from pathlib import Path

RAW_DIR = Path("../data/raw")
for p in sorted(RAW_DIR.iterdir()):
    if p.is_file():
        print(f"{p.name:35s} {p.stat().st_size / 1e6:>10.1f} MB")

Intermediary Data.XLSX                     0.7 MB
README.md                                  0.0 MB
Respondent Data.XLSX                     171.6 MB


In [3]:
from python_calamine import CalamineWorkbook

RESPONDENT_FILE = RAW_DIR / "Respondent Data.XLSX"
INTERMEDIARY_FILE = RAW_DIR / "Intermediary Data.XLSX"

for f in (RESPONDENT_FILE, INTERMEDIARY_FILE):
    wb = CalamineWorkbook.from_path(str(f))
    print(f.name, "-> sheets:", wb.sheet_names)

Respondent Data.XLSX -> sheets: ['RLD']
Intermediary Data.XLSX -> sheets: ['T1']


Each workbook has a single sheet: `T1` (Intermediary) and `RLD` (Respondent). Both follow the same layout convention: **row 1 = short column codes, row 2 = the full question text/description, row 3 onward = data** — confirmed below.

### What the intermediary workbook contains (documented briefly, then left aside)

`Intermediary Data.XLSX` (sheet `T1`) is **1,313 records × 70 columns** (excluding the two header rows). It surveys **market intermediaries** (RIAs, distributors, brokers) — e.g. `Q002_VCW`: "Which category do you belong to", `Q003_VCX`: years in the intermediary service, `Q005_VCZ`: current customer base size — and their *perception* of investor familiarity with, and risk-perception of, various securities products (`GridXQ006_VBB`, `GridXQ007_VBC` grids: one row per product — Stocks, Mutual Funds, F&O, ETF/Gold ETF, Corporate Bonds, REITs/InvIT, AIF).

This is **intermediary-reported opinion data, not respondent-level investor data** — a different unit of analysis from the main study population. It is not used further in this notebook. One data-quality note for whoever revisits it later: its identifier column `INTNR` has 7 duplicated values out of 1,313 rows (1,306 unique) — not investigated here.

The rest of this notebook works only with the **respondent workbook** (`RLD` sheet), which is the main SEBI Investor Survey 2025 population.

In [4]:
int_wb = CalamineWorkbook.from_path(str(INTERMEDIARY_FILE))
int_sheet = int_wb.get_sheet_by_name(int_wb.sheet_names[0])
int_data = int_sheet.to_python(skip_empty_area=True)
int_codes, int_descs, int_rows = int_data[0], int_data[1], int_data[2:]

int_df = pd.DataFrame(int_rows, columns=int_codes)
print("Intermediary records x columns:", int_df.shape)
print("INTNR missing:", int_df["INTNR"].isna().sum())
print("INTNR duplicated (non-null):", int_df["INTNR"].duplicated().sum())
print("INTNR distinct:", int_df["INTNR"].nunique())

# free memory: not used further
del int_wb, int_sheet, int_data

Intermediary records x columns: (1313, 70)
INTNR missing: 0
INTNR duplicated (non-null): 7
INTNR distinct: 1306


## 2. Identify the structure of the respondent workbook

Load the `RLD` sheet and look at the first few rows *before* assuming which row holds column names. Only the two header rows are printed below (codes and question descriptions) — these are not respondent data, so they are safe to display.

In [5]:
resp_wb = CalamineWorkbook.from_path(str(RESPONDENT_FILE))
resp_sheet = resp_wb.get_sheet_by_name(resp_wb.sheet_names[0])
resp_data = resp_sheet.to_python(skip_empty_area=True)  # ~10-15s for ~109k x 448

codes = resp_data[0]
descriptions = resp_data[1]

print("Row 1 sample (column codes):          ", codes[:6])
print("Row 2 sample (question descriptions): ", descriptions[:6])

def is_uninformative(code, desc):
    d = (desc or "").strip()
    return d in ("", code, code + ":")

bare = [c for c, d in zip(codes, descriptions) if is_uninformative(c, d)]
print(f"\n{len(bare)} of {len(codes)} columns have a row-2 'description' that adds no text beyond the code itself, e.g.: {bare[:8]}")
print("For these, the workbook simply does not provide a fuller description — none is invented for the data dictionary.")

Row 1 sample (column codes):           ['Resp_ID_DP', 'UniqueId_DP', 'SELECTED_STATE', 'URBANRURAL', 'Q1', 'SECNEW']
Row 2 sample (question descriptions):  ['Resp_ID_DP', 'UniqueId_DP', 'Selected_State:', 'UrbanRural: Urban-Rural Classification', 'Q1: Gender', 'SECNEW: NCCS']

5 of 448 columns have a row-2 'description' that adds no text beyond the code itself, e.g.: ['Resp_ID_DP', 'UniqueId_DP', 'Zone_DP', 'Life_Stage', 'Q13M']
For these, the workbook simply does not provide a fuller description — none is invented for the data dictionary.


In [6]:
from collections import Counter

resp_rows = resp_data[2:]  # data starts after the two header rows

dup_codes = {k: v for k, v in Counter(codes).items() if v > 1}
print("Duplicate column codes:", dup_codes if dup_codes else "none")

n_records = len(resp_rows)
n_columns = len(codes)
print(f"\nRespondent records (excluding the 2 header rows): {n_records:,}")
print(f"Columns: {n_columns}")

Duplicate column codes: none

Respondent records (excluding the 2 header rows): 109,430
Columns: 448


In [7]:
df = pd.DataFrame(resp_rows, columns=codes)
desc_of = dict(zip(codes, descriptions))  # column code -> question description, kept alongside df
print("df shape:", df.shape)

df shape: (109430, 448)


### Survey stages / incomplete surveys

The workbook is **not** a single flat pass over uniform questions — there are two stages, and only some respondents reach the second one.

In [8]:
for col in ["MAIN_COMP_STATUS", "QLISTMAIN", "QFL", "INT_TYPE"]:
    print(f"=== {col}  ({desc_of[col]}) ===")
    print(df[col].value_counts(dropna=False))
    print()

=== MAIN_COMP_STATUS  (MAIN COMPLETE) ===
MAIN_COMP_STATUS
                 56073
Main Complete    53357
Name: count, dtype: int64

=== QLISTMAIN  (QLISTMAIN: QLISTMAIN. Please select the applicable option based on the respondent’s willingness and qualification) ===
QLISTMAIN
Close the interview with the Listing section only     56073
Continue with the Mains section right now with MEP    53357
Name: count, dtype: int64

=== QFL  (QFL: Final Investor â€“ Non-Investor Classification) ===
QFL
NON-INVESTOR    81548
INVESTOR        27882
Name: count, dtype: int64

=== INT_TYPE  (INT_TYPE: &lt;font style='font-weight:bold'&gt; Type of Interview &lt;/font&gt;) ===
INT_TYPE
Random     91950
Booster    17480
Name: count, dtype: int64



**Reading this:** every one of the 109,430 recruited respondents went through a **Listing** (screening) stage. `QLISTMAIN` records the routing decision taken at the end of Listing: 56,073 respondents were closed out with the Listing section only, and 53,357 continued into the **Mains** (detailed) section — which matches `MAIN_COMP_STATUS` exactly (blank vs. `"Main Complete"`). `QFL` (Investor / Non-Investor classification, 27,882 / 81,548) is a *different* split, decided during Listing, and does not line up with the Mains-completion split — so the two must not be conflated. `INT_TYPE` shows the sample also mixes a `Random` main sample (91,950) with a `Booster` sample (17,480), which matters for weighting.

This is a strong signal, confirmed in Section 5, that many columns (nearly all "M"-suffixed questions and the per-product grids) are only populated for the 53,357 Mains-completers, and some are populated for narrower sub-groups still. Blank there does not mean "No" — it means the question was never put to that respondent.

## 3. Document the columns

Build `data/processed/data_dictionary.csv` with one row per column: the code, the row-2 description (verbatim — nothing invented for columns where the workbook has none), an observed data type, non-missing/missing counts and percentage, and the number of distinct non-missing values.

**Missingness convention used here:** `python_calamine` returns an empty string `''` for a blank cell (not `NaN`), including for otherwise-numeric columns. A cell is counted as missing if it is `NaN` **or** an empty string after stripping. This is a routing/non-response signal, not a substantive answer — explicit responses such as *"Don't Know / Can't Say"*, *"Can't remember"*, *"Do not wish to disclose"*, *"None of the above"* are kept as their own distinct values, not folded into "missing" (see Section 5).

In [9]:
def classify_kind(v):
    if isinstance(v, str):
        return "text"
    if isinstance(v, (int, float)):
        return "numeric"
    return type(v).__name__

records = []
for code in codes:
    s = df[code]
    is_blank = s.isna() | (s.astype(object) == "")
    non_missing = s[~is_blank]
    n_non_missing = len(non_missing)
    n_missing = n_records - n_non_missing
    uniq_vals = non_missing.unique()
    kinds = sorted(set(classify_kind(v) for v in uniq_vals))
    if kinds == ["numeric"]:
        obs_dtype = "numeric"
    elif kinds == ["text"]:
        obs_dtype = "text"
    elif not kinds:
        obs_dtype = "all_blank"
    else:
        obs_dtype = "mixed(" + "+".join(kinds) + ")"
    records.append({
        "column_code": code,
        "question_description": desc_of[code],
        "observed_dtype": obs_dtype,
        "non_missing_count": n_non_missing,
        "missing_count": n_missing,
        "missing_pct": round(100 * n_missing / n_records, 2),
        "n_distinct_non_missing": len(uniq_vals),
    })

data_dictionary = pd.DataFrame(records)
print(data_dictionary.shape)
data_dictionary["observed_dtype"].value_counts()

(448, 7)


observed_dtype
text                   241
all_blank              134
numeric                 52
mixed(numeric+text)     21
Name: count, dtype: int64

In [10]:
out_path = Path("../data/processed/data_dictionary.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
data_dictionary.to_csv(out_path, index=False)
print("wrote", out_path.resolve(), "-", len(data_dictionary), "rows")

all_blank_cols = data_dictionary.loc[data_dictionary["observed_dtype"] == "all_blank", "column_code"].tolist()
print(f"\n{len(all_blank_cols)} columns are entirely blank across all {n_records:,} records (0 non-missing values).")
print("First 10:", all_blank_cols[:10])

wrote /Users/keshavdubey/Downloads/Work/Monsoon 2026/Market Research & Validation/Mutual Funds Analysis/data/processed/data_dictionary.csv - 448 rows

134 columns are entirely blank across all 109,430 records (0 non-missing values).
First 10: ['GRIDxP1[{_5}].P1', 'GRIDxP1[{_6}].P1', 'GRIDxP1[{_8}].P1', 'GRIDxP1[{_9}].P1', 'GRIDxP1[{_10}].P1', 'GRIDxP1[{_12}].P1', 'GRIDxP1[{_13}].P1', 'GRIDxP1[{_14}].P1', 'GRIDxP1[{_15}].P1', 'GRIDxP1[{_16}].P1']


**All-blank columns, unresolved:** 134 of 448 columns (about 30%) have zero non-missing values across all 109,430 records. Nearly all of them are per-product slots inside the repeated grid questions (`GRIDxP1`, `GridxQ7`, `Q14M_RANK_GRID`, `Q15M_RANK_GRID`, `Q4_Q5_Inv_Filt`/`Q4_Q5_NONInv_Filt`, `ADI_Dashboard`, `Q2MXGrid`, `Q2M_DP_Filt`) for products **other than** Futures & Options, Stocks/Shares, REITs/InvIT, Corporate Bonds, AIF, and the combined MF+ETF slot (`{_1_2}`) — e.g. every NPS, Gold ETF, Chit Fund, PPF/VPF, SGB, EPF, PMS, Gold-physical, Crypto and SIF slot in these particular grids is entirely empty. Whether this reflects a genuinely zero incidence for those products in this fielding, or a narrower export/routing that only ever populated a focus set of products for these specific grids, is **not established from the file alone** — flagged as an open question in `docs/data_inspection.md` rather than guessed at here. These columns are kept in the dictionary and not dropped.

**Encoding artifact, also from the source file itself (not introduced by our reading tools):** several description and answer-option strings contain mojibake such as `â€“` where an en-dash (`–`) or curly punctuation should be. This was verified by reading `xl/sharedStrings.xml` directly from the `.xlsx` zip archive — the double-encoded UTF-8 bytes are present in the original workbook. It affects display of certain punctuation only, not the substance of codes or values.

## 4. Fields relevant to the study

Candidate columns identified from the codes/descriptions above, grouped by topic. Values and frequencies are inspected below for each before treating any of them as final.

In [11]:
CANDIDATE_FIELDS = {
    "Survey completion / stage": ["MAIN_COMP_STATUS", "QLISTMAIN", "INT_TYPE", "QFL"],
    "Age / generation": ["Life_Stage"],
    "Employment / occupation": ["Q14", "CWE"],
    "Income": ["Q10", "Q10A"],
    "Investment awareness": ["Q21A", "Q1A", "Q29"],
    "Mutual-fund consideration": ["Q23A", "Q25A", "AA1_DD1", "AA2_DD2", "AA3_DD3"],
    "Current mutual-fund holdings": [
        "Q22A_All", "GRIDxP1[{_1_2}].P1", "Q2MXGrid[{_1_2}].Q2M", "ADI_Dashboard[{_1_2}].Slice",
    ],
    "Previous mutual-fund investment": ["Q24A", "NI_Dashboard[{_1_2}].Slice", "A15_D15", "AA4_DD4"],
    "Reported investment barriers": ["A13_D13", "A14_D14", "AA2_DD2", "AA3_DD3"],
    "Survey weights": ["WeightMainM2", "Weight_to_Sample"],
    "Respondent identifier": ["Resp_ID_DP", "UniqueId_DP"],
}

for topic, cols in CANDIDATE_FIELDS.items():
    print(f"\n### {topic}")
    for c in cols:
        print(f"  {c:30s} {desc_of.get(c, '')[:100]}")


### Survey completion / stage
  MAIN_COMP_STATUS               MAIN COMPLETE
  QLISTMAIN                      QLISTMAIN: QLISTMAIN. Please select the applicable option based on the respondent’s willingness and 
  INT_TYPE                       INT_TYPE: &lt;font style='font-weight:bold'&gt; Type of Interview &lt;/font&gt;
  QFL                            QFL: Final Investor â€“ Non-Investor Classification

### Age / generation
  Life_Stage                     Life_Stage

### Employment / occupation
  Q14                            Q14: Q14. What is your current primary occupation?
  CWE                            CWE: CWE. And can you now tell me who is the chief wage earner in the family i.e., the one who contr

### Income
  Q10                            Q10: &lt;font color='Red'&gt; SHOW SCREEN TO RESPONDENT &lt;/font&gt; Q10. Among the following broad
  Q10A                           Q10a: &lt;font color='Red'&gt; SHOW SCREEN TO RESPONDENT &lt;/font&gt; Q10a. And among the followi

**Note on age:** no raw "age in years" question was found anywhere among the 448 columns. The closest available field is `Life_Stage`, a derived generational classification (Gen Z / Millennials / Generation X / Baby Boomers) with no separate description row — its exact construction (birth-year cutoffs) is not stated in the workbook and should not be assumed.

Survey completion/stage (`MAIN_COMP_STATUS`, `QLISTMAIN`, `QFL`, `INT_TYPE`) value counts were already inspected in Section 2 above.

### Age / generation, employment, income

In [12]:
for col in ["Life_Stage", "Q14", "Q10", "Q10A"]:
    print(f"=== {col}  ({desc_of[col]}) ===")
    print(df[col].value_counts(dropna=False).head(15))
    print()

=== Life_Stage  (Life_Stage) ===
Life_Stage
Gen Z           49035
Millennials     45235
Generation X    12806
Baby Boomers     2354
Name: count, dtype: int64

=== Q14  (Q14: Q14. What is your current primary occupation?) ===
Q14
Homemaker / Housewife                                                                     14219
Student                                                                                   11512
Skilled worker like electrician/Mechanic etc.                                             10848
Shop Owner- operate from a permanent establishment e.g. wholesalers, distributors etc.     8574
Clerk / Salesman                                                                           7414
Petty trader- street vendor, drivers owning vehicles etc.                                  6993
Supervisory Level                                                                          5573
Businessmen/Industrialist with no employees under him/her                                  4803
Own

### Investment awareness, Demat/trading account

In [13]:
for col in ["Q29", "Q1A"]:
    s = df[col]
    is_blank = s.isna() | (s == "")
    print(f"=== {col}  ({desc_of[col]}) ===")
    print(f"blank/not-administered: {is_blank.sum():,} of {len(s):,}")
    print(s[~is_blank].value_counts().head(10))
    print()

# Q21A is multi-select (comma-separated); report top single-product mentions rather than raw combinations
q21a_exploded = df.loc[df["Q21A"] != "", "Q21A"].str.split(",").explode().str.strip()
print(f"=== Q21A  ({desc_of['Q21A']}) — individual products mentioned, top 15 ===")
print(q21a_exploded.value_counts().head(15))

=== Q29  (Q29: Do you have a Demat account / Share Market trading account?) ===
blank/not-administered: 0 of 109,430
Q29
No                                         82606
Yes                                        22413
Can’t remember (DO NOT AID THIS OPTION)     4411
Name: count, dtype: int64

=== Q1A  (Q1A: Are you aware of the entities that are directly involved in regulating or operating the securities market in India?) ===
blank/not-administered: 56,073 of 109,430
Q1A
Reserve Bank of India (RBI)                                                                                                                                                                                                                                                 14904
None of the above                                                                                                                                                                                                                                        

=== Q21A  (Q21a: Which of the following financial product(s) are you aware of?) — individual products mentioned, top 15 ===
Q21A
Fixed Deposits / Recurring Deposit / Bank Savings Account                             108446
Life insurance / Unit Linked Insurance Plans (ULIPS)                                  106835
Gold - Physical form / Sovereign Gold Bond (SGB)                                       85271
Post office savings / Kisan Vikas Patra (KVP) / National Savings Certificate (NSC)     82982
MF_ETF                                                                                 73316
Mutual Funds (One-time Lumpsum / SIP)                                                  72299
Stocks / Shares                                                                        67852
Real Estate as an Investment (excluding where you are staying)                         64989
Chit Fund                                                                              40995
National Pension System (NPS)     

**Parsing hazard found here:** naively splitting these multi-select strings on `,` is not safe — one answer option's own label contains a comma: `"Cryptocurrency (e.g. Tether, Bitcoin, Ethereum, etc)"`. The `.str.split(",")` used above (a first look only) incorrectly shreds this single option into fake separate "products" (`Cryptocurrency (e.g. Tether`, `Bitcoin`, `Ethereum, etc)` truncated further) — visible in the Q21A/Q22A_All counts below. Any later notebook that explodes these multi-select fields for real must split on the known option vocabulary, not on a bare comma.

### Current mutual-fund holdings, previous investment, future consideration, barriers

`Q22A_All` (currently hold), `Q24A` (ever invested in the past) and `Q23A`/`Q25A` (future consideration / will-never-consider) are all multi-select fields listing specific products — **Mutual Funds** and **ETF/Gold ETF** are kept as distinct list items here, with an additional combined `MF_ETF` tag appended when either is present (see the dedicated note below). The `A#_D#`/`AA#_DD#`/`ADI_Dashboard`/`GRIDxP1[{_1_2}]` family of columns, by contrast, exist **only** at the combined MF+ETF level — there is no way to isolate plain Mutual Funds from ETF/Gold ETF in those follow-up questions.

In [14]:
def top_products(col, n=15):
    s = df[col]
    blank = (s == "")
    print(f"=== {col}  ({desc_of[col]}) ===")
    print(f"blank/not-administered: {blank.sum():,} of {len(s):,}")
    exploded = s[~blank].str.split(",").explode().str.strip()
    print(exploded.value_counts().head(n))
    print()

for col in ["Q22A_All", "Q24A", "Q23A", "Q25A"]:
    top_products(col)

=== Q22A_All  (Q22A_All: Which of the following financial products do you currently hold investments in) ===
blank/not-administered: 0 of 109,430


Q22A_All
Fixed Deposits / Recurring Deposit / Bank Savings Account                             51203
None of the above                                                                     36016
Life insurance / Unit Linked Insurance Plans (ULIPS)                                  25904
MF+ETF                                                                                18624
Mutual Funds (One-time Lumpsum / SIP)                                                 18328
Gold - Physical form / Sovereign Gold Bond (SGB)                                      14156
Stocks / Shares                                                                       13751
Post office savings / Kisan Vikas Patra (KVP) / National Savings Certificate (NSC)    12677
Chit Fund                                                                              4007
Real Estate as an Investment (excluding where you are staying)                         2428
Employees Provident Fund (EPF)                                         

Q23A
None of the above                                                                         27801
MF_ETF                                                                                     5962
Mutual Funds (One-time Lumpsum / SIP)                                                      5205
Stocks / Shares                                                                            3950
Futures & Options (F&O)                                                                     949
Exchange Trade Funds (ETF) / Gold Exchange Trade Funds (Gold ETF)                           927
Real Estate Investment Trusts (REITs) and /or Infrastructure Investment Trusts (InvIT)      913
Corporate Bonds                                                                             528
Alternate Investment Fund (AIF)                                                             258
Name: count, dtype: int64

=== Q25A  (Q25a: Which of the following financial products will you never consider investing in the futu

In [15]:
# MF+ETF-only follow-up fields: holdings recency/share, active/dormant status, and reported barriers
for col in [
    "GRIDxP1[{_1_2}].P1", "Q2MXGrid[{_1_2}].Q2M", "ADI_Dashboard[{_1_2}].Slice", "NI_Dashboard[{_1_2}].Slice",
    "A11_D11", "A12_D12", "A13_D13", "A14_D14", "A15_D15",
    "AA1_DD1", "AA2_DD2", "AA3_DD3", "AA4_DD4",
]:
    s = df[col]
    blank = (s == "")
    print(f"{col:35s} blank: {blank.sum():>7,} / {len(s):,}   ({desc_of[col][:70]})")

GRIDxP1[{_1_2}].P1                  blank:  95,309 / 109,430   (MF_ETF : For each of the financial product(s) that you currently hold )
Q2MXGrid[{_1_2}].Q2M                blank:  95,309 / 109,430   (MF+ETF : Q2M_1: TOTAL)
ADI_Dashboard[{_1_2}].Slice         blank:  95,309 / 109,430   (MF_ETF : Active/Dormant Investor Based Q22C and productwise for dashbo)
NI_Dashboard[{_1_2}].Slice          blank:  56,354 / 109,430   (MF_ETF : NI_Dashboard: Non- Investor and productwise for dashboard)
A11_D11                             blank:  95,568 / 109,430   (A11_D11:MF+ETF - Frequently you invest in MF/ETF.)
A12_D12                             blank:  95,568 / 109,430   (A12_D12: MF+ETF - What you think are the expected returns for Mutual F)
A13_D13                             blank:  95,568 / 109,430   (A13_D13:MF+ETF - What challenges do you face before/ while making fres)
A14_D14                             blank:  95,568 / 109,430   (A14_D14:  What challenges do you face after making investm

In [16]:
# Barrier fields are multi-select "top 3 reasons" (kept as raw combinations here — a later
# notebook can decide whether to explode them into individual reason flags)
for col in ["A13_D13", "A14_D14", "AA2_DD2", "AA3_DD3"]:
    s = df[col]
    non_blank = s[s != ""]
    print(f"=== {col}  ({desc_of[col]}) — top combinations among the {len(non_blank):,} non-blank answers ===")
    print(non_blank.value_counts().head(5))
    print()

=== A13_D13  (A13_D13:MF+ETF - What challenges do you face before/ while making fresh investment in MF/ETF) — top combinations among the 13,862 non-blank answers ===
A13_D13
Uncertainty about returns and performance,Fear of significant losses due to market volatility,Lack of trust in the mutual funds             341
Uncertainty about returns and performance,Fear of significant losses due to market volatility,Regulatory uncertainties or policy changes    243
Fear of significant losses due to market volatility,Regulatory uncertainties or policy changes,Lack of trust in the mutual funds            213
Uncertainty about returns and performance,Regulatory uncertainties or policy changes,Lack of trust in the mutual funds                      210
Lack of trust in the mutual funds,Difficulty in making payment/moving funds,High fees, charges and Management expenses                      186
Name: count, dtype: int64

=== A14_D14  (A14_D14:  What challenges do you face after making investment in 

### Survey weights and respondent identifiers

In [17]:
for col in ["WeightMainM2", "Weight_to_Sample"]:
    s = df[col]
    blank = (s == "")
    numeric = pd.to_numeric(s[~blank], errors="coerce")
    print(f"=== {col}  ({desc_of[col]}) ===")
    print(f"blank: {blank.sum():,} / {len(s):,}")
    print(numeric.describe())
    print()

for col in ["Resp_ID_DP", "UniqueId_DP"]:
    s = df[col]
    print(f"{col}: missing={s.isna().sum()}, duplicated={s.duplicated().sum()}, distinct={s.nunique()}, n={len(s)}")

=== WeightMainM2  (Weight (Group 2)(Main)) ===
blank: 56,073 / 109,430
count    53357.000000
mean         1.000000
std          1.063544
min          0.011126
25%          0.235019
50%          0.668440
75%          1.292676
max          5.392610
Name: WeightMainM2, dtype: float64

=== Weight_to_Sample  (Weight (Sample)) ===
blank: 0 / 109,430
count    109430.000000
mean          1.000000
std           1.091765
min           0.008273
25%           0.269063
50%           0.759417
75%           1.083635
max           6.879864
Name: Weight_to_Sample, dtype: float64

Resp_ID_DP: missing=0, duplicated=0, distinct=109430, n=109430
UniqueId_DP: missing=0, duplicated=0, distinct=109430, n=109430


### Do any questions combine Mutual Funds and ETFs?

Yes, both patterns exist in this workbook and must be told apart when scoping "mutual funds" for this study:

- **Kept separate, with a combined tag added:** `Q21A` (awareness), `Q22A_All` (current holdings), `Q23A`/`Q25A` (future consideration / will never consider), `Q24A` (past investment) list `"Mutual Funds (One-time Lumpsum / SIP)"` and `"Exchange Trade Funds (ETF) / Gold Exchange Trade Funds (Gold ETF)"` as separate selectable items, and *also* append an `MF_ETF`/`MF+ETF` marker to the same multi-select string whenever either was chosen. A plain-MF count and a plain-ETF count can both be recovered from these fields.
- **Combined only, no way to separate:** every `A#_D#` / `AA#_DD#` column, `GRIDxP1[{_1_2}]`, `Q2MXGrid[{_1_2}]`, `GridxQ7[{_1_2}]`, `Q14M_RANK_GRID[{_1_2}]`, `Q15M_RANK_GRID[{_1_2}]`, `Q4_Q5_Inv_Filt[{_1_2}]`/`Q4_Q5_NONInv_Filt[{_1_2}]`, and both dashboard slices ask about **"MF+ETF" as a single combined product** — reasons for/against investing, barriers, frequency, expected returns, and the active/dormant classification are only ever recorded at the combined level. There is no column in this workbook that reports these follow-up details for Mutual Funds alone, excluding ETFs.

Any analysis of "reasons/barriers for mutual funds" will therefore actually be reporting on mutual funds **and** ETFs/Gold ETFs together, not mutual funds in isolation — this needs to be stated explicitly wherever those columns are used.

## 5. Interpretation risks

### Blank vs. explicit non-response

Blanks (routing skips) and explicit substantive answers must be counted separately — folding "Don't Know" or "Not applicable" into a blank, or a blank into "No", would misrepresent both.

In [18]:
EXPLICIT_NONRESPONSE_MARKERS = [
    "Don’t Know / Can’t Say", "Can’t remember (DO NOT AID THIS OPTION)",
    "Do not wish to disclose (DO NOT AID THIS OPTION)", "No current income (DO NOT AID THIS OPTION)",
    "Choose not to answer (DO NOT AID THIS OPTION)",
]
# Note: "None of the above" is deliberately NOT treated as non-response here — for fields like
# Q22A_All it is a substantive answer ("holds none of these products"), not a refusal/DK marker,
# and should not be conflated with the two categories below.

for col in ["Q1A", "Q29", "Q10", "Q10A", "Q13"]:
    s = df[col]
    blank = (s == "")
    explicit_nonresponse = s.isin(EXPLICIT_NONRESPONSE_MARKERS)
    print(f"{col:12s} blank(routing skip)={blank.sum():>7,}   explicit DK/refusal marker={explicit_nonresponse.sum():>7,}   substantive answer={(~blank & ~explicit_nonresponse).sum():>7,}")

Q1A          blank(routing skip)= 56,073   explicit DK/refusal marker=  2,575   substantive answer= 50,782


Q29          blank(routing skip)=      0   explicit DK/refusal marker=  4,411   substantive answer=105,019
Q10          blank(routing skip)=      0   explicit DK/refusal marker=  8,244   substantive answer=101,186
Q10A         blank(routing skip)=      0   explicit DK/refusal marker= 24,488   substantive answer= 84,942
Q13          blank(routing skip)=      0   explicit DK/refusal marker=    210   substantive answer=109,220


### Different respondent groups answered different questions

Section 2 already showed 56,073 respondents were closed out at Listing and never saw the Mains section. Below, that is checked directly against a Mains-only field (`Q1A`), and then routing is shown to narrow further *within* Mains — the MF+ETF-specific follow-ups (`A13_D13`) are answered by a smaller subgroup still, presumably gated on holding or having considered MF/ETF specifically.

In [19]:
listing_only = df["MAIN_COMP_STATUS"] == ""
q1a_blank = df["Q1A"] == ""
a13_blank = df["A13_D13"] == ""

print("Listing-only respondents:                 ", listing_only.sum())
print("Q1A blank (Mains-only question):           ", q1a_blank.sum())
print("Do the two sets match exactly?             ", (listing_only == q1a_blank).all())
print()
print("A13_D13 blank (MF+ETF-specific follow-up): ", a13_blank.sum(), "-- narrower than Mains-only blank, i.e. gated further within Mains")
print("A13_D13 blank among Mains-completers only: ", a13_blank[~listing_only].sum(), "of", (~listing_only).sum(), "Mains completers")

Listing-only respondents:                  56073
Q1A blank (Mains-only question):            56073
Do the two sets match exactly?              True

A13_D13 blank (MF+ETF-specific follow-up):  95568 -- narrower than Mains-only blank, i.e. gated further within Mains
A13_D13 blank among Mains-completers only:  39495 of 53357 Mains completers


### Duplicate / missing respondent IDs (checked, not removed)

Already computed above under "Survey weights and respondent identifiers": both `Resp_ID_DP` and `UniqueId_DP` have **0 missing and 0 duplicated values** across all 109,430 records (109,430 distinct values each). No records are removed here regardless of the outcome — this is a check only.

### Remaining open questions (not resolved by inspecting the file alone)

- Whether the 134 all-blank columns reflect true zero incidence or a narrower export (Section 3).
- The exact construction of `Life_Stage` (age-band cutoffs are not stated in the workbook).
- The precise routing logic that gates `A#_D#`/`AA#_DD#` MF+ETF follow-ups within Mains-completers (observed empirically above, not documented in a codebook).
- Whether `WeightMainM2` (Mains-only) or `Weight_to_Sample` (full listing sample) is the appropriate weight for a given analysis — both exist and serve different populations.
- The intermediary workbook's 7 duplicate `INTNR` values (noted, not investigated — that workbook is out of scope here).

## 6. Summary

This notebook only **inspects** the respondent workbook — no filtering of the research sample, no imputation, no calculated findings, and no dashboard charts. `data/processed/data_dictionary.csv` (respondent-level, gitignored) and `docs/data_inspection.md` (structure/quality summary, no respondent-level examples, safe to commit) are the two outputs. See `docs/data_inspection.md` for the write-up, and the candidate column mapping + open questions below for what needs deciding before any sample is filtered.